# 05 — mmWave Studio Connection Control

This notebook demonstrates the Python API workflow for establishing a
connection to the AWR2944EVM through mmWave Studio.

**Architecture:** Python/Jupyter is the control layer. mmWave Studio is
the execution backend. Python automatically executes Lua scripts in
mmWave Studio — no manual Lua paste required.

## Prerequisites

```bash
python -m pip install -e ".[automation]"
```

In [ ]:
from pathlib import Path
from awr2944_dca.api.experiment import Experiment

## 1. Open Experiment and Save COM Port

In [ ]:
exp = Experiment.open('.')
print(f'Experiment root: {exp.root_dir}')

In [ ]:
from awr2944_dca.hardware.ports import scan_ports, save_local_hardware

# Scan and display available ports
ports = scan_ports()
for p in ports:
    print(f'{p.com:8s} | {p.friendly_name:40s} | role={p.likely_role} ({p.confidence})')

In [ ]:
# Save your COM port (change to your actual port)
chosen_com = 'COM6'
cfg_path = save_local_hardware(exp.root_dir, 'awr-rs232', chosen_com)
print(f'Saved to {cfg_path.name}')

## 2. Check mmWave Studio Status

In [ ]:
from awr2944_dca.mmws.executor import detect_available_modes

modes = detect_available_modes()
for m in modes:
    status = 'AVAILABLE' if m.available else 'not available'
    print(f'{m.mode.value:25s} | {status:15s} | {m.detail}')

## 3. Smoke Test (no hardware)

Verify Python → mmWave Studio → Lua → JSON → Python works automatically.

In [ ]:
from awr2944_dca.mmws.bridge import StudioBridge, StageStatus

log_dir = exp.root_dir / 'ti' / 'probe_logs'
bridge = StudioBridge(log_dir)

# Generate and execute smoke test
script = bridge.generate_smoke_script()
result = bridge.execute(script, mode='auto', timeout=30)

print(f'Status: {result["status"].value}')
print(f'Transport: {result["exec_result"].mode.value}')
if result['stage_result']:
    print(f'WriteToLog available: {result["stage_result"].get("log_available")}')

## 4. Generate and Execute Connection Script

This is the main event — Python generates the connection Lua,
automatically executes it in mmWave Studio, waits for the result JSON,
and reports connection status.

In [ ]:
from awr2944_dca.mmws.models import ConnectionTabConfig

config = ConnectionTabConfig(rs232_com=chosen_com)
script = bridge.generate_connection_script(config.com_number)

# Auto-execute in mmWave Studio
result = bridge.execute(script, mode='auto', timeout=30)

print(f'Execution transport: {result["exec_result"].mode.value}')
print(f'Status: {result["status"].value}')

if result['stage_result']:
    sr = result['stage_result']
    print(f'COM: {sr.get("com_display")}')
    print(f'Connect return: {sr.get("connect_return")}')
    print(f'Error: {sr.get("error")}')

## 5. Check Connection Status (from files)

In [ ]:
from awr2944_dca.mmws.stages import StageName

status, data = bridge.check_status(StageName.CONNECTION_ONLY)
print(f'Status: {status.value}')
if data:
    print(f'  COM: {data.get("com_display")}')
    print(f'  Connect return: {data.get("connect_return")}')